# sim2spec Perlmutter Bootcamp

This notebook mirrors `ProjectReadMe.md` in a shorter day-by-day format for new HPC users on NERSC Perlmutter.

Important conventions used below:

- `WORKDIR` means the project repo root: `$PSCRATCH/HPC_intro/sim2spec`
- `setup.sh` defines the common paths and virtual environment name
- GPU simulation cells use `srun` when the notebook is not already running on a compute node

**Prefer batch jobs?** Every GPU step also has a corresponding sbatch script in `scripts/`. For example:
```bash
sbatch scripts/sbatch_day1_smoke.sh
sbatch scripts/sbatch_day2_baseline.sh
sbatch scripts/sbatch_day4_sweep.sh
sbatch scripts/sbatch_day5_profile_baseline.sh
sbatch scripts/sbatch_day5_profile_compare.sh
```
Remember to replace `<your_account>` with your NERSC project account before submitting.

## Before You Begin: Allocation Notes

Do **not** run `salloc` as a notebook cell. `salloc` is interactive and belongs in a terminal. In a notebook cell it can appear to hang because it is waiting to hold an interactive allocation.

If you want a terminal allocation, run this in a terminal:

```bash
salloc -C gpu -q interactive -t 00:30:00 -A <YOUR_ACCOUNT>
```

Then run commands in that same terminal, or start your Jupyter server/kernel from inside that allocation.

In this notebook, GPU work is launched with `srun`. If the notebook is already on a compute node whose hostname starts with `nid`, the cells run directly. Otherwise they request a short GPU job for the command being run.


In [ ]:
import socket
print(socket.gethostname())
!squeue -u $USER


# Day 1: Environment Setup, Install, and Smoke Test

Goal: copy the project to scratch, create the Python environment, install `sim2spec` and `larnd-sim`, then run a minimal GPU smoke test.


In [ ]:
%%bash
# Clone the project to scratch. Safe to rerun.
export SCRATCH_ROOT=$PSCRATCH/HPC_intro
export WORKDIR=$SCRATCH_ROOT/sim2spec

mkdir -p "$SCRATCH_ROOT"
cd "$SCRATCH_ROOT"
git clone https://github.com/madantimalsina/sim2spec.git

cd "$WORKDIR"
pwd
ls

In [ ]:
%%bash
# Install the project environment. This can take several minutes the first time.
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
cd "$WORKDIR"

source setup.sh
bash install.sh


In [ ]:
%%bash
# Validate Python packages that do not require running a simulation.
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
cd "$WORKDIR"

source setup.sh
source "$venv_name/bin/activate"

python -c "import fire; print('fire ok')"
python -c "import larndsim; print('larndsim ok')"
#sim2spec --help | head


In [ ]:
%%bash
# Validate GPU/CuPy. Uses srun unless this notebook is already on a compute node.
export ACCOUNT=${ACCOUNT:-YOUR_ACCOUNT}   # replace with your NERSC project account

if [[ $(hostname) == nid* ]]; then
    bash -lc '
        export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
        cd "$WORKDIR"
        export MPLBACKEND=Agg
        source setup.sh
        source "$venv_name/bin/activate"
        hostname
        python -c "import cupy as cp; print(int(cp.arange(10).sum()))"
    '
else
    srun -A "$ACCOUNT" -q interactive -C gpu --gpus=1 --ntasks=1 --cpus-per-task=8 -t 00:30:00 bash -lc '
        export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
        cd "$WORKDIR"
        export MPLBACKEND=Agg
        source setup.sh
        source "$venv_name/bin/activate"
        hostname
        python -c "import cupy as cp; print(int(cp.arange(10).sum()))"
    '
fi

In [ ]:
%%bash
# Day 1 smoke test. This is intentionally tiny: one event.
export ACCOUNT=${ACCOUNT:-YOUR_ACCOUNT}   # replace with your NERSC project account

run_smoke='\
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec; \
cd "$WORKDIR"; \
export MPLBACKEND=Agg; \
source setup.sh; \
source "$venv_name/bin/activate"; \
export LARNDSIM_DISABLE_CUPY_MEMPOOL=1; \
mkdir -p "$OUTBASE"; \
hostname; \
sim2spec run --larndsim-dir "$LARNDSIM_DIR" --config 2x2 --input "$INPUT_H5" --outdir "$OUTBASE/day1_smoke" --n-events 1; \
sim2spec qa --run-dir "$OUTBASE/day1_smoke/run"\
'

if [[ $(hostname) == nid* ]]; then
    bash -lc "$run_smoke"
else
    srun -A "$ACCOUNT" -q interactive -C gpu --gpus=1 --ntasks=1 --cpus-per-task=8 -t 00:30:00 bash -lc "$run_smoke"
fi

In [ ]:
%%bash
# Check Day 1 outputs.
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
cd "$WORKDIR"
source setup.sh

run_dir="$OUTBASE/day1_smoke/run"
metrics="$run_dir/qa/metrics.json"

if [[ ! -d "$run_dir" ]]; then
    echo "Missing $run_dir"
    echo "Rerun the Day 1 smoke-test cell above first."
    exit 1
fi

ls -R "$run_dir"
echo
if [[ -f "$metrics" ]]; then
    head "$metrics"
else
    echo "Missing $metrics"
    echo "The smoke test created the run directory but QA did not finish."
    exit 1
fi


# Day 2: Baseline Run and First Output Validation

Goal: run a small baseline simulation and inspect the generated output, manifest, and QA metrics.


In [ ]:
%%bash
# Day 2 baseline run. Increase --n-events later if the small run works.
export ACCOUNT=${ACCOUNT:-YOUR_ACCOUNT}   # replace with your NERSC project account

run_day2='\
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec; \
cd "$WORKDIR"; \
export MPLBACKEND=Agg; \
source setup.sh; \
source "$venv_name/bin/activate"; \
export LARNDSIM_DISABLE_CUPY_MEMPOOL=1; \
mkdir -p "$OUTBASE"; \
hostname; \
sim2spec run --larndsim-dir "$LARNDSIM_DIR" --config 2x2 --input "$INPUT_H5" --outdir "$OUTBASE/day2_baseline" --n-events 5; \
sim2spec qa --run-dir "$OUTBASE/day2_baseline/run"\
'

if [[ $(hostname) == nid* ]]; then
    bash -lc "$run_day2"
else
    srun -A "$ACCOUNT" -q interactive -C gpu --gpus=1 --ntasks=1 --cpus-per-task=8 -t 00:45:00 bash -lc "$run_day2"
fi

In [ ]:
%%bash
# Inspect Day 2 outputs.
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
cd "$WORKDIR"
source setup.sh

ls -R "$OUTBASE/day2_baseline/run"
echo
cat "$OUTBASE/day2_baseline/run/manifest.json"
echo
cat "$OUTBASE/day2_baseline/run/qa/metrics.json"


In [ ]:
%%bash
# Save Day 2 QA metrics as CSV.
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
cd "$WORKDIR"
source setup.sh
source "$venv_name/bin/activate"

python - <<'PY'
import csv, json, os
path = os.environ["OUTBASE"] + "/day2_baseline/run/qa/metrics.json"
out = os.environ["OUTBASE"] + "/day2_baseline/run/qa/metrics.csv"
d = json.load(open(path))
keys = sorted(d)
with open(out, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=keys)
    writer.writeheader()
    writer.writerow(d)
print(f"Saved: {out}")
PY


# Day 3: Baseline QA Interpretation and Simple Analysis

Goal: read the Day 2 baseline `output.h5`, save Day 3 analysis products in a separate folder, and connect the output to physical interpretation.


In [ ]:
%%bash
# Create validation plots from the Day 2 baseline output.
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
export MPLBACKEND=Agg
cd "$WORKDIR"
source setup.sh
source "$venv_name/bin/activate"

python - <<'PY'
import os
import h5py
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

path = os.environ["OUTBASE"] + "/day2_baseline/run/output.h5"
out_dir = os.environ["OUTBASE"] + "/day3_analysis/run"
os.makedirs(out_dir, exist_ok=True)

with h5py.File(path, "r") as f:
    packets = f["packets"][:]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(packets["timestamp"].astype(float), packets["dataword"].astype(float), s=2, alpha=0.4)
    ax.set_xlabel("Timestamp [ticks]")
    ax.set_ylabel("Charge [ADC counts]")
    ax.set_title("Charge vs. Time")
    fig.tight_layout()
    fig.savefig(f"{out_dir}/plot_charge_vs_time.png", dpi=150)

    segments = f["segments"][:]
    event_id, count = np.unique(segments["event_id"], return_counts=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(event_id, count, width=0.6)
    ax.set_xlabel("Event ID")
    ax.set_ylabel("Number of Segments")
    ax.set_title("Hits per Event")
    fig.tight_layout()
    fig.savefig(f"{out_dir}/plot_hits_per_event.png", dpi=150)

    if "light_wvfm" in f:
        wvfm = f["light_wvfm"][0, 0, :]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(wvfm, lw=1)
        ax.set_xlabel("Sample index")
        ax.set_ylabel("ADC counts")
        ax.set_title("Single Light Waveform")
        fig.tight_layout()
        fig.savefig(f"{out_dir}/plot_single_waveform.png", dpi=150)

print(f"Saved plots in {out_dir}")
PY

find "$OUTBASE/day3_analysis/run" -maxdepth 1 -name 'plot_*.png' | sort


In [ ]:
%%bash
# Optional: run the project plotting helper.
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
export MPLBACKEND=Agg
cd "$WORKDIR"
source setup.sh
source "$venv_name/bin/activate"

mkdir -p "$OUTBASE/day3_analysis/run"
ln -sfn "$OUTBASE/day2_baseline/run/output.h5" "$OUTBASE/day3_analysis/run/output.h5"
python plot_validation.py "$OUTBASE/day3_analysis/run/output.h5"


# Day 4: Parameter Sweeps and Provenance Tracking

Goal: run a small sweep, check that each variant produced metrics, and save a comparison CSV.

The sweep runs four variants defined in `configs/sweep.yaml`, each using a different random seed (42, 1337, 55555, 99999). Because the simulation has stochastic components, each variant will produce observably different packet counts and ADC distributions. Each run is saved with its full provenance — seed used, git commit, environment — so results can be reproduced and compared systematically.

The output directories will be named `000_seed_42`, `001_seed_1337`, `002_seed_55555`, and `003_seed_99999`.

In [ ]:
%%bash
# Day 4 sweep.
export ACCOUNT=${ACCOUNT:-YOUR_ACCOUNT}   # replace with your NERSC project account

run_day4='\
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec; \
cd "$WORKDIR"; \
export MPLBACKEND=Agg; \
source setup.sh; \
source "$venv_name/bin/activate"; \
export LARNDSIM_DISABLE_CUPY_MEMPOOL=1; \
hostname; \
sim2spec sweep --larndsim-dir "$LARNDSIM_DIR" --config 2x2 --input "$INPUT_H5" --outdir "$OUTBASE/day4_sweep" --sweep "$SIM2SPEC_DIR/configs/sweep.yaml" --n-events 3\
'

if [[ $(hostname) == nid* ]]; then
    bash -lc "$run_day4"
else
    srun -A "$ACCOUNT" -q interactive -C gpu --gpus=1 --ntasks=1 --cpus-per-task=8 -t 01:00:00 bash -lc "$run_day4"
fi

In [ ]:
%%bash
# Inspect sweep outputs and save comparison CSV.
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
cd "$WORKDIR"
source setup.sh
source "$venv_name/bin/activate"

ls "$OUTBASE/day4_sweep"
find "$OUTBASE/day4_sweep" -maxdepth 2 -name manifest.json
find "$OUTBASE/day4_sweep" -maxdepth 3 -name metrics.json

python - <<'PY'
import csv, glob, json, os

root = os.environ["OUTBASE"] + "/day4_sweep"
out = root + "/comparison.csv"
rows = []

for mpath in sorted(glob.glob(root + "/*/qa/metrics.json")):
    variant = mpath.split("/")[-3]
    run_dir = os.path.dirname(os.path.dirname(mpath))
    manifest_path = os.path.join(run_dir, "manifest.json")
    metrics = json.load(open(mpath))
    manifest = json.load(open(manifest_path)) if os.path.exists(manifest_path) else {}

    rows.append({
        "variant": variant,
        "config": manifest.get("larndsim", {}).get("config"),
        "seed": manifest.get("sim", {}).get("rand_seed"),
        "n_events": manifest.get("sim", {}).get("n_events"),
        "git_commit": manifest.get("larndsim", {}).get("git", {}).get("commit"),
        "n_packets": metrics.get("n_packets"),
        "adc_mean": metrics.get("adc_mean", metrics.get("adc_mean_guess")),
        "adc_std": metrics.get("adc_std", metrics.get("adc_std_guess")),
        "n_light_wvfm": metrics.get("n_light_wvfm"),
    })

fields = ["variant", "config", "seed", "n_events", "git_commit", "n_packets", "adc_mean", "adc_std", "n_light_wvfm"]
with open(out, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

for row in rows:
    print(row)
print(f"Saved: {out}")
PY


# Day 5: Profiling and One Measurable Improvement

Goal: profile a baseline run, make one simple comparison run, and record enough information to compare the results.


In [ ]:
%%bash
# Day 5 baseline profile with Nsight Systems.
export ACCOUNT=${ACCOUNT:-YOUR_ACCOUNT}   # replace with your NERSC project account

run_day5_base='\
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec; \
cd "$WORKDIR"; \
export MPLBACKEND=Agg; \
source setup.sh; \
source "$venv_name/bin/activate"; \
export LARNDSIM_DISABLE_CUPY_MEMPOOL=1; \
hostname; \
sim2spec run --larndsim-dir "$LARNDSIM_DIR" --config 2x2 --input "$INPUT_H5" --outdir "$OUTBASE/day5_profile_baseline" --n-events 10 --profiler nsys; \
sim2spec profile --run-dir "$OUTBASE/day5_profile_baseline/run"\
'

if [[ $(hostname) == nid* ]]; then
    bash -lc "$run_day5_base"
else
    srun -A "$ACCOUNT" -q interactive -C gpu --gpus=1 --ntasks=1 --cpus-per-task=8 -t 01:00:00 bash -lc "$run_day5_base"
fi

### Before running the comparison profile

You need to make one code change to `larnd-sim` to create a meaningful comparison. Open:

```
larnd-sim/cli/simulate_pixels.py
```

Go to **line 1280** and change:
```python
TPB = 4
```
to:
```python
TPB = 128
```

This changes the CUDA thread-block size. `TPB = 4` is far below a GPU warp (32 threads), so the GPU is very underutilised. `TPB = 128` is a more standard setting. Both runs use `--n-events 10` so the wall-time comparison is apple-to-apple.

In [ ]:
%%bash
# Day 5 comparison profile.
# Before running this cell, change TPB from 4 to 128 in:
#   larnd-sim/cli/simulate_pixels.py  (line 1280)
export ACCOUNT=${ACCOUNT:-YOUR_ACCOUNT}   # replace with your NERSC project account

run_day5_compare='\
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec; \
cd "$WORKDIR"; \
export MPLBACKEND=Agg; \
source setup.sh; \
source "$venv_name/bin/activate"; \
export LARNDSIM_DISABLE_CUPY_MEMPOOL=1; \
hostname; \
sim2spec run --larndsim-dir "$LARNDSIM_DIR" --config 2x2 --input "$INPUT_H5" --outdir "$OUTBASE/day5_profile_compare" --n-events 10 --profiler nsys; \
sim2spec profile --run-dir "$OUTBASE/day5_profile_compare/run"\
'

if [[ $(hostname) == nid* ]]; then
    bash -lc "$run_day5_compare"
else
    srun -A "$ACCOUNT" -q interactive -C gpu --gpus=1 --ntasks=1 --cpus-per-task=8 -t 01:00:00 bash -lc "$run_day5_compare"
fi

In [ ]:
%%bash
# Compare profiling artifacts and save basic system information.
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
cd "$WORKDIR"
source setup.sh
source "$venv_name/bin/activate"

cat "$OUTBASE/day5_profile_baseline/run/profile/nsys_stats.json" | head -n 40

python - <<'PY'
import os
base = os.environ["OUTBASE"]
for name in ["day5_profile_baseline", "day5_profile_compare"]:
    run = f"{base}/{name}/run"
    manifest = f"{run}/manifest.json"
    output = f"{run}/output.h5"
    print(f"\n{name}")
    if os.path.exists(manifest) and os.path.exists(output):
        print("  approx_wall_seconds:", round(os.path.getmtime(output) - os.path.getmtime(manifest), 2))
        print("  output_size_MB:", round(os.path.getsize(output) / (1024 * 1024), 2))
    else:
        print("  missing manifest or output.h5")
PY


# Final Cross-Day Summary

Goal: collect the main metrics and artifacts needed for a short final presentation.


In [ ]:
%%bash
export WORKDIR=$PSCRATCH/HPC_intro/sim2spec
cd "$WORKDIR"
source setup.sh
source "$venv_name/bin/activate"

python - <<'PY'
import glob, json, os
base = os.environ["OUTBASE"]

baseline = f"{base}/day2_baseline/run/qa/metrics.json"
print("=== BASELINE ===")
if os.path.exists(baseline):
    d = json.load(open(baseline))
    print("baseline", d.get("n_packets"), d.get("adc_mean", d.get("adc_mean_guess")))
else:
    print("missing baseline metrics")

print("\n=== DAY 4 SWEEP ===")
for path in sorted(glob.glob(f"{base}/day4_sweep/*/qa/metrics.json")):
    d = json.load(open(path))
    print(path.split("/")[-3], d.get("n_packets"), d.get("adc_mean", d.get("adc_mean_guess")))
PY

echo
find "$OUTBASE" -maxdepth 4 \( -name "metrics.json" -o -name "manifest.json" -o -name "*.png" -o -name "nsys_stats.json" \) | sort


## Suggested Final Presentation Structure

1. Day 1: environment and install validated
2. Day 2-3: baseline output, QA metrics, and validation plots
3. Day 4: controlled sweep and provenance
4. Day 5: profiling results and one measured comparison
